## Theoretical Considerations — Statistical Validation (Minimum Sample Size)

Assume we have verified ground-truth sales data (true total revenue) and/or verified match labels for a subset of locations. We want a minimum sample size that supports statistically meaningful validation.

---

### 1) Validating Matching Quality (Entity Resolution)

If we treat matching as a classification problem (correct vs incorrect match), we can estimate the proportion of correct matches $p$ and require the margin of error $E$ at confidence level $(1-\alpha)$.

A standard sample size formula for estimating a proportion is:

$$
n = \frac{z_{\alpha/2}^2 \, p(1-p)}{E^2}
$$

Where:

- $z_{\alpha/2}$ = z-score for the confidence level (e.g., 1.96 for 95%)
- $p$ = expected accuracy (use pilot estimate; if unknown use 0.5 for worst-case)
- $E$ = desired absolute error (e.g., ±2% or ±3%)

**Parameters this depends on:**

- Desired confidence level ($z$)
- Expected match accuracy $p$
- Acceptable error margin $E$
- (Optional) finite population correction if sampling without replacement from a limited population

**Practical note:**  
Because match behavior can vary by geography and business type, a stratified sample is better:

- Stratify by `match_status` (match vs possible)
- Optionally by state or `sqft_bucket`

Then allocate enough samples per stratum to estimate accuracy per segment.

---

### 2) Validating Revenue Adjustment (Multiplier Model)

If we have verified true total revenue $T$ for sampled locations/time-windows, and our model produces adjusted revenue $\hat{T}$, we can validate the **mean error**:

$$
e_i = \hat{T}_i - T_i
$$

To estimate the mean error $\mu_e$ within margin $E$ at confidence level $(1-\alpha)$, sample size can be approximated by:

$$
n \approx \left(\frac{z_{\alpha/2}\,\sigma_e}{E}\right)^2
$$

Where:

- $\sigma_e$ = standard deviation of errors (can be estimated from a pilot)
- $E$ = acceptable error bound for mean error (e.g., ±$X or ±Y%)

Alternatively, if validating MAPE (percentage error), define:

$$
\text{APE}_i = \left|\frac{\hat{T}_i - T_i}{T_i}\right|
$$

and use the same form with $\sigma_{\text{APE}}$.

**Parameters this depends on:**

- Confidence level ($z$)
- Variability of error ($\sigma_e$ or $\sigma_{\text{APE}}$)
- Desired precision ($E$)
- Heterogeneity across segments (entity / sqft / state) → motivates stratified sampling

**Practical note:**  
Stratify the validation set by `business_entity_id` (or entity size) and `sqft_bucket`, since payment mix can differ by chain and store format.

## Theoretical Considerations — Model Comparison

Assume we are comparing multiple revenue adjustment models (e.g., Model 1 constant multiplier vs Model 2 segmented multiplier) using a subset of locations with verified total revenue.

---

### Metrics to Evaluate Superiority

If ground-truth total revenue $T$ is available:

**Pointwise error metrics**

- MAE:  
  $$
  \frac{1}{n}\sum_{i=1}^n \left| \hat{T}_i - T_i \right|
  $$

- RMSE (penalizes large errors):  
  $$
  \sqrt{\frac{1}{n}\sum_{i=1}^n (\hat{T}_i - T_i)^2}
  $$

- MAPE / sMAPE (percentage-based metrics, useful across different revenue scales)

- Bias (mean error):  
  $$
  \frac{1}{n}\sum_{i=1}^n (\hat{T}_i - T_i)
  $$

---

**Stability / robustness**

- Error by segment: MAE / MAPE per `business_entity_id`, `sqft_bucket`, state
- Outlier rate: % of records above the 99th percentile of absolute error

---

If ground-truth is **not** available (as in this test), compare models via plausibility and stability checks:

- Distribution sanity: adjusted revenue distribution should shift smoothly from raw card revenue without extreme inflation
- Revenue/day/sqft sanity: compare adjusted revenue per day per sqft by bucket; check for unrealistic tails
- Model deviation: distribution of multiplier ratios  
  $$
  \frac{\text{Model 2 multiplier}}{\text{Model 1 multiplier}}
  $$
  should be bounded and explainable
- Within-entity stability: multiplier variance within the same `business_entity_id` should not be excessive unless justified

---

### How to Ensure Comparison Validity

To compare models fairly:

1. Use the **same matched subset** of records for all models (same `raw_id`s and time windows).
2. Use the **same evaluation metric definitions** and the same filtering rules (e.g., remove invalid `days_in_window`).
3. If splitting train/validation (for any tuned parameters), ensure splits are:
   - Stratified by entity and size segment
   - Time-consistent if temporal leakage matters
4. Prefer **paired comparisons** on the same records:
   - Compute per-record error difference:
     $$
     |e^{(A)}_i| - |e^{(B)}_i|
     $$
   - Summarize mean difference and confidence interval
5. Report performance by strata (entity / sqft / state) to ensure one model isn’t “better on average” but worse for key segments.

---

**Interpretation:**  
A model is superior if it reduces error (or improves plausibility/stability without ground truth) consistently across key segments, not just on overall averages.

## Chosen Adjustment Method (Summary)

- **Model 1 (constant multiplier)** is the simplest baseline and provides an interpretable benchmark.
- **Model 2 (segmented + bounded + shrunk)** is preferred as the final approach because it allows card-share to vary by store segment (`business_entity_id × sqft_bucket`) while remaining stable through shrinkage and explicit bounds on card share.

In the absence of true total revenue, I selected Model 2 because it produces reasonable multiplier dispersion, reduces extreme outliers slightly relative to the baseline, and remains explainable and auditable.